# NexaMem with Azure Cache for Redis - Key Operations Demo

This notebook demonstrates how to use the **NexaMem** package with **Azure Cache for Redis** for managing chatbot memory and conversation history using key-based operations.

## What You'll Learn:
- Setting up NexaMem with Azure Cache for Redis
- Storing and retrieving memory data using keys
- Working with memory scopes and channels
- Managing key expiry and deletion
- Using both synchronous and asynchronous APIs

## Prerequisites:
- Azure Cache for Redis instance
- Redis access key or Entra ID authentication
- Python 3.8+ environment

## 1. Install and Import Required Libraries

First, let's install the required packages and import the necessary libraries.

In [ ]:
# Install required packages (run this if packages are not already installed)
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import nexamem
    print("✓ NexaMem is already installed")
except ImportError:
    print("Installing NexaMem...")
    install_package("nexamem")

try:
    import redis
    print("✓ Redis-py is already installed")
except ImportError:
    print("Installing redis-py...")
    install_package("redis")

print("All packages are ready!")

In [ ]:
# Import required libraries
import asyncio
import os
import uuid
from typing import Dict, Any, Optional, List
from datetime import datetime, timedelta

# Import NexaMem components
from nexamem import (
    AIMemory,
    AsyncAIMemory,
    AIMemoryConfig,
    AzureRedisConfig,
    StorageConfig,
    MemoryScope
)

print("Successfully imported all NexaMem components!")
print("Available classes:")
print("- AIMemory: Synchronous memory management")
print("- AsyncAIMemory: Asynchronous memory management")
print("- AIMemoryConfig: Configuration management")
print("- AzureRedisConfig: Azure Redis configuration")
print("- StorageConfig: Storage backend configuration")
print("- MemoryScope: Memory scope management")

## 2. Configure Azure Cache for Redis Connection

Now let's set up the configuration for connecting to Azure Cache for Redis. You'll need to replace the placeholder values with your actual Azure Redis instance details.

In [ ]:
# Azure Cache for Redis Configuration
# IMPORTANT: Replace these values with your actual Azure Redis instance details

# Method 1: Direct configuration
AZURE_REDIS_HOSTNAME = "your-redis-cache.redis.cache.windows.net"
AZURE_REDIS_ACCESS_KEY = "your-primary-or-secondary-access-key"
AZURE_REDIS_PORT = 6380  # Default SSL port for Azure Redis
AZURE_REDIS_DB = 0

# Method 2: Environment variables (recommended for production)
# You can set these environment variables instead:
# export AZURE_REDIS_HOSTNAME="your-redis-cache.redis.cache.windows.net"
# export AZURE_REDIS_ACCESS_KEY="your-access-key"

# Use environment variables if available, otherwise fall back to direct values
redis_hostname = os.getenv('AZURE_REDIS_HOSTNAME', AZURE_REDIS_HOSTNAME)
redis_access_key = os.getenv('AZURE_REDIS_ACCESS_KEY', AZURE_REDIS_ACCESS_KEY)
redis_port = int(os.getenv('AZURE_REDIS_PORT', str(AZURE_REDIS_PORT)))
redis_db = int(os.getenv('AZURE_REDIS_DB', str(AZURE_REDIS_DB)))

print(f"Redis Configuration:")
print(f"  Hostname: {redis_hostname}")
print(f"  Port: {redis_port}")
print(f"  Database: {redis_db}")
print(f"  SSL: Enabled (required for Azure)")
print(f"  Access Key: {'*' * (len(redis_access_key) - 4) + redis_access_key[-4:] if redis_access_key else 'Not set'}")

In [ ]:
# Create NexaMem configuration for Azure Cache for Redis
def create_azure_redis_config() -> AIMemoryConfig:
    """Create configuration for Azure Cache for Redis."""
    
    # Configure Azure Redis connection
    azure_redis_config = AzureRedisConfig(
        hostname=redis_hostname,
        port=redis_port,
        access_key=redis_access_key,
        db=redis_db,
        ssl=True,  # Required for Azure
        socket_timeout=5.0,
        socket_connect_timeout=5.0,
        max_connections=50,
        retry_on_timeout=True,
        use_collections=True,  # Use Redis collections for better organization
        collection_prefix="nexamem"  # Prefix for all Redis keys
    )
    
    # Create storage configuration
    storage_config = StorageConfig.azure_redis(azure_redis_config)
    
    # Create main AIMemory configuration
    config = AIMemoryConfig(
        default_scope="chatbot_session",  # Default scope for memory operations
        storage=storage_config,
        enable_audit=True,  # Enable audit logging
        enable_metrics=True,  # Enable metrics collection
        async_preferred=True  # Prefer async operations when available
    )
    
    return config

# Create the configuration
config = create_azure_redis_config()
print("✓ NexaMem configuration created successfully!")
print(f"  Default Scope: {config.default_scope}")
print(f"  Storage Type: {config.storage.backend}")
print(f"  Audit Enabled: {config.enable_audit}")
print(f"  Metrics Enabled: {config.enable_metrics}")

## 3. Connect to Azure Cache for Redis

Now let's establish connections to Azure Cache for Redis using both the synchronous and asynchronous NexaMem APIs.

In [ ]:
# Create synchronous and asynchronous NexaMem instances
try:
    # Create synchronous AIMemory instance
    memory = AIMemory(config)
    print("✓ Synchronous AIMemory instance created successfully!")
    
    # Create asynchronous AIMemory instance
    async_memory = AsyncAIMemory(config)
    print("✓ Asynchronous AIMemory instance created successfully!")
    
    # Test the connection by getting storage stats
    stats = memory.get_stats()
    print(f"\n📊 Storage Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
    
    print("\n🎉 Successfully connected to Azure Cache for Redis!")
    
except Exception as e:
    print(f"❌ Failed to connect to Azure Cache for Redis: {e}")
    print("Please check your configuration and ensure your Redis instance is accessible.")

## 4. Set and Get Keys Using NexaMem API

Let's demonstrate how to store and retrieve memory data using the NexaMem API with Azure Cache for Redis. NexaMem provides high-level memory management operations that map to Redis keys internally.

In [ ]:
# Example 1: Store and retrieve simple memory data
print("=== Basic Memory Operations ===\n")

# Store a simple conversation memory
conversation_content = "User asked about Python programming, I provided basic syntax examples."
metadata = {
    "user_id": "user123",
    "timestamp": datetime.now().isoformat(),
    "topic": "python_programming",
    "confidence": 0.95
}

# Store the memory (NexaMem will auto-generate a memory ID)
memory_id = memory.store(
    content=conversation_content,
    metadata=metadata,
    scope="python_tutorial"
)

print(f"✓ Stored memory with ID: {memory_id}")
print(f"  Content: {conversation_content}")
print(f"  Scope: python_tutorial")
print(f"  Metadata keys: {list(metadata.keys())}")

# Retrieve the memory
retrieved_memory = memory.retrieve(memory_id)

if retrieved_memory:
    print(f"\n✓ Retrieved memory successfully:")
    print(f"  ID: {retrieved_memory.get('memory_id')}")
    print(f"  Content: {retrieved_memory.get('content')}")
    print(f"  Scope: {retrieved_memory.get('scope')}")
    print(f"  Created: {retrieved_memory.get('created_at')}")
    print(f"  Metadata: {retrieved_memory.get('metadata')}")
else:
    print("❌ Failed to retrieve memory")

In [ ]:
# Example 2: Store multiple memories and demonstrate search
print("\n=== Multiple Memories and Search ===\n")

# Store multiple conversation memories
conversations = [
    {
        "content": "User asked about machine learning algorithms, I explained supervised vs unsupervised learning.",
        "metadata": {"topic": "machine_learning", "user_id": "user123", "difficulty": "beginner"}
    },
    {
        "content": "User requested help with data visualization in Python using matplotlib.",
        "metadata": {"topic": "data_visualization", "user_id": "user456", "difficulty": "intermediate"}
    },
    {
        "content": "Discussion about web development with Flask framework and REST APIs.",
        "metadata": {"topic": "web_development", "user_id": "user789", "difficulty": "advanced"}
    }
]

memory_ids = []
for i, conv in enumerate(conversations):
    memory_id = memory.store(
        content=conv["content"],
        metadata=conv["metadata"],
        scope="tutorial_sessions"
    )
    memory_ids.append(memory_id)
    print(f"✓ Stored conversation {i+1}: {memory_id[:8]}...")

# List all memories in the scope
print(f"\n📋 Listing all memories in 'tutorial_sessions' scope:")
all_memories = memory.list(scope="tutorial_sessions")
print(f"Found {len(all_memories)} memories")
for mem_id in all_memories:
    print(f"  - {mem_id}")

# Search for specific content
print(f"\n🔍 Searching for 'Python' related memories:")
search_results = memory.search(query="Python", scope="tutorial_sessions")
print(f"Found {len(search_results)} matches:")
for result_id in search_results:
    result = memory.retrieve(result_id)
    if result:
        print(f"  - {result_id[:8]}...: {result['content'][:50]}...")

In [ ]:
# Example 3: Async operations for better performance
print("\n=== Asynchronous Operations ===\n")

async def demo_async_operations():
    """Demonstrate asynchronous memory operations."""
    
    # Store memory asynchronously
    async_content = "Async conversation about concurrent programming and asyncio in Python."
    async_metadata = {
        "type": "async_demo",
        "timestamp": datetime.now().isoformat(),
        "async": True
    }
    
    async_memory_id = await async_memory.store(
        content=async_content,
        metadata=async_metadata,
        scope="async_sessions"
    )
    
    print(f"✓ Async stored memory: {async_memory_id}")
    
    # Retrieve memory asynchronously
    async_retrieved = await async_memory.retrieve(async_memory_id)
    print(f"✓ Async retrieved memory: {async_retrieved['content'][:50]}...")
    
    # List memories asynchronously
    async_memories = await async_memory.list(scope="async_sessions")
    print(f"✓ Found {len(async_memories)} memories in async_sessions scope")
    
    return async_memory_id

# Run the async demo
async_result = await demo_async_operations()

## 5. Working with Key Expiry and Deletion

NexaMem provides powerful capabilities for managing memory lifecycle, including deletion and working with channel-based TTL (Time-To-Live) configurations.

In [ ]:
# Example 1: Individual memory deletion
print("=== Memory Deletion Operations ===\n")

# Create a temporary memory for deletion demo
temp_content = "This is a temporary memory that will be deleted."
temp_memory_id = memory.store(
    content=temp_content,
    metadata={"temporary": True},
    scope="temp_scope"
)

print(f"✓ Created temporary memory: {temp_memory_id}")

# Verify it exists
temp_memory = memory.retrieve(temp_memory_id)
print(f"✓ Memory exists: {temp_memory is not None}")

# Delete the memory
deletion_success = memory.delete(temp_memory_id)
print(f"✓ Deletion successful: {deletion_success}")

# Verify it's gone
deleted_memory = memory.retrieve(temp_memory_id)
print(f"✓ Memory after deletion: {deleted_memory is None}")

# Example 2: Bulk deletion by scope
print(f"\n=== Bulk Deletion by Scope ===\n")

# Count memories before deletion
before_count = len(memory.list(scope="tutorial_sessions"))
print(f"Memories in 'tutorial_sessions' before deletion: {before_count}")

# Clear entire scope
deleted_count = memory.clear_scope("tutorial_sessions")
print(f"✓ Deleted {deleted_count} memories from scope")

# Count memories after deletion
after_count = len(memory.list(scope="tutorial_sessions"))
print(f"Memories in 'tutorial_sessions' after deletion: {after_count}")

In [ ]:
# Example 3: Working with Channels and TTL
print("\n=== Channels and TTL Management ===\n")

# Register channels with different TTL settings
print("Registering channels with TTL settings:")

# Short-term channel (5 minutes)
memory.register_channel(
    name="short_term",
    ttl_sec=300,  # 5 minutes
    encrypt=False
)
print("✓ Registered 'short_term' channel (5 minutes TTL)")

# Medium-term channel (1 hour)
memory.register_channel(
    name="medium_term", 
    ttl_sec=3600,  # 1 hour
    encrypt=False
)
print("✓ Registered 'medium_term' channel (1 hour TTL)")

# Long-term channel (24 hours)
memory.register_channel(
    name="long_term",
    ttl_sec=86400,  # 24 hours
    encrypt=True,  # Enable encryption for sensitive data
    quota_bytes=1048576  # 1MB quota
)
print("✓ Registered 'long_term' channel (24 hours TTL, encrypted)")

# Get channel information
print(f"\nChannel configurations:")
for channel_name in ["short_term", "medium_term", "long_term"]:
    channel = memory.get_channel(channel_name)
    if channel:
        print(f"  {channel_name}:")
        print(f"    TTL: {channel.ttl_sec} seconds")
        print(f"    Encrypted: {channel.encrypt}")
        if hasattr(channel, 'quota_bytes') and channel.quota_bytes:
            print(f"    Quota: {channel.quota_bytes} bytes")

# Store memories in different channels (TTL will be applied automatically)
channels_demo = [
    ("short_term", "Quick note: User prefers dark mode"),
    ("medium_term", "Session preference: User is working on a Python project"),
    ("long_term", "User profile: Experienced developer, prefers detailed explanations")
]

for channel_name, content in channels_demo:
    channel_memory_id = memory.store(
        content=content,
        metadata={"channel": channel_name, "demo": True},
        scope=f"{channel_name}_scope"
    )
    print(f"✓ Stored in {channel_name}: {channel_memory_id[:8]}...")

## 6. Audit Logging and Monitoring

NexaMem provides comprehensive audit logging and monitoring capabilities to track memory operations and system performance.

In [ ]:
# Audit Logging and System Statistics
print("=== Audit Logging and Monitoring ===\n")

# Get audit log entries
print("📋 Recent audit log entries:")
audit_entries = memory.get_audit_log(limit=10)

if audit_entries:
    for i, entry in enumerate(audit_entries[:5], 1):  # Show first 5 entries
        print(f"  {i}. {entry.get('timestamp', 'N/A')} - {entry.get('action', 'N/A')}")
        if entry.get('memory_id'):
            print(f"     Memory ID: {entry['memory_id'][:8]}...")
        if entry.get('details'):
            print(f"     Details: {entry['details']}")
        print()
else:
    print("  No audit entries found")

# Get system statistics and performance metrics
print("📊 System Statistics:")
stats = memory.get_stats()

for key, value in stats.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:,}")
    else:
        print(f"  {key}: {value}")

# Memory usage by scope
print(f"\n📈 Memory Distribution by Scope:")
scopes = ["python_tutorial", "async_sessions", "short_term_scope", "medium_term_scope", "long_term_scope"]

for scope in scopes:
    memories = memory.list(scope=scope)
    if memories:
        print(f"  {scope}: {len(memories)} memories")
        # Show sample memory from this scope
        sample = memory.retrieve(memories[0])
        if sample:
            content_preview = sample['content'][:40] + "..." if len(sample['content']) > 40 else sample['content']
            print(f"    Sample: {content_preview}")

print(f"\n🎯 Total unique scopes with data: {len([s for s in scopes if memory.list(scope=s)])}")

## 7. Advanced Features and Best Practices

Let's explore some advanced features and best practices for production use of NexaMem with Azure Cache for Redis.

In [ ]:
# Advanced Features Demo
print("=== Advanced Features and Best Practices ===\n")

# 1. Using Memory Scopes for isolation
print("1. 🏷️  Memory Scope Management:")

# Create memories in different scopes for different users/sessions
users = ["alice", "bob", "charlie"]
for user in users:
    user_scope = f"user_{user}_session"
    memory_id = memory.store(
        content=f"User {user} is working on a data science project with pandas and numpy.",
        metadata={
            "user": user,
            "session_start": datetime.now().isoformat(),
            "project_type": "data_science"
        },
        scope=user_scope
    )
    print(f"  ✓ Created memory for {user} in scope '{user_scope}'")

# 2. Batch operations for efficiency
print(f"\n2. ⚡ Batch Operations:")

batch_data = [
    ("Conversation about API design patterns", {"topic": "architecture"}),
    ("Discussion on database optimization", {"topic": "database"}),
    ("Code review feedback on error handling", {"topic": "code_quality"})
]

batch_ids = []
for content, metadata in batch_data:
    memory_id = memory.store(content=content, metadata=metadata, scope="batch_demo")
    batch_ids.append(memory_id)

print(f"  ✓ Stored {len(batch_ids)} memories in batch")

# Retrieve all batch memories
print("  📋 Batch retrieval results:")
for i, memory_id in enumerate(batch_ids, 1):
    mem = memory.retrieve(memory_id)
    if mem:
        print(f"    {i}. {mem['metadata']['topic']}: {mem['content'][:30]}...")

# 3. Error handling and resilience
print(f"\n3. 🛡️  Error Handling:")

try:
    # Attempt to retrieve non-existent memory
    fake_id = str(uuid.uuid4())
    result = memory.retrieve(fake_id)
    print(f"  ✓ Graceful handling of missing memory: {result is None}")
    
    # Attempt to delete non-existent memory
    delete_result = memory.delete(fake_id)
    print(f"  ✓ Graceful handling of missing deletion: {not delete_result}")
    
except Exception as e:
    print(f"  ❌ Unexpected error: {e}")

# 4. Memory with custom IDs
print(f"\n4. 🎯 Custom Memory IDs:")

custom_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
memory.store(
    content="Session summary: Covered advanced Python topics including decorators and context managers.",
    memory_id=custom_id,
    metadata={"session_type": "advanced_python", "duration_minutes": 45},
    scope="educational_sessions"
)

retrieved_custom = memory.retrieve(custom_id)
print(f"  ✓ Stored and retrieved memory with custom ID: {custom_id}")
print(f"  Content: {retrieved_custom['content'][:50]}...")

## 8. Production Tips and Configuration

Here are some important considerations and best practices for using NexaMem with Azure Cache for Redis in production environments.

In [ ]:
# Production Configuration and Best Practices
print("=== Production Configuration Guide ===\n")

print("🏭 Production Configuration Recommendations:")
print("""
1. 🔐 Security:
   - Use environment variables for sensitive data (access keys, connection strings)
   - Enable SSL/TLS (always enabled for Azure Cache for Redis)
   - Consider using Azure Entra ID authentication instead of access keys
   - Enable encryption for sensitive memory channels

2. 🔧 Performance:
   - Set appropriate connection pool sizes (50-100 for high-traffic apps)
   - Use async operations for I/O-intensive workloads
   - Configure appropriate socket timeouts (5-10 seconds)
   - Enable Redis collections for better key organization

3. 📊 Monitoring:
   - Enable audit logging for compliance
   - Monitor memory usage and Redis metrics
   - Set up alerts for connection failures
   - Track memory operation latencies

4. 💾 Data Management:
   - Use appropriate TTL settings for different data types
   - Implement memory quotas for channels
   - Regular cleanup of old scopes
   - Consider data backup strategies

5. 🌐 High Availability:
   - Use Azure Cache for Redis Premium tier for clustering
   - Configure retry logic for transient failures
   - Implement circuit breaker patterns
   - Consider geographic replication for global apps
""")

# Example production configuration
print("📋 Example Production Configuration:")

production_config_example = '''
# Environment Variables (set in your deployment environment)
export AZURE_REDIS_HOSTNAME="prod-cache.redis.cache.windows.net"
export AZURE_REDIS_ACCESS_KEY="your-production-access-key"
export AZURE_REDIS_PORT="6380"
export AZURE_REDIS_DB="0"

# Optional: Advanced configuration
export NEXAMEM_DEFAULT_SCOPE="production"
export NEXAMEM_ENABLE_AUDIT="true"
export NEXAMEM_ENABLE_METRICS="true"
export NEXAMEM_STRICT_YAML_VALIDATION="true"
'''

print(production_config_example)

# Show environment-based configuration
print("🔧 Loading Configuration from Environment:")
try:
    env_config = AIMemoryConfig.from_env()
    print(f"  ✓ Default Scope: {env_config.default_scope}")
    print(f"  ✓ Audit Enabled: {env_config.enable_audit}")
    print(f"  ✓ Metrics Enabled: {env_config.enable_metrics}")
    print(f"  ✓ Storage Backend: {env_config.storage.backend}")
except Exception as e:
    print(f"  ⚠️  Could not load from environment: {e}")
    print("  This is expected in demo mode - set environment variables for production")

## 9. Summary and Next Steps

Congratulations! You've successfully learned how to use NexaMem with Azure Cache for Redis for managing chatbot memory and conversation history.

### What we covered:
- ✅ Setting up NexaMem with Azure Cache for Redis
- ✅ Storing and retrieving memory data using keys  
- ✅ Working with memory scopes and channels
- ✅ Managing key expiry and deletion
- ✅ Using both sync and async APIs
- ✅ Audit logging and monitoring
- ✅ Advanced features and production best practices

### Key Benefits of NexaMem + Azure Redis:
- **Scalability**: Handle millions of conversations with Redis clustering
- **Performance**: Sub-millisecond latency for memory operations
- **Reliability**: Built-in persistence and high availability
- **Security**: SSL encryption and Azure security features
- **Flexibility**: Multiple authentication methods and configuration options

### Next Steps:
1. **Integrate with your chatbot**: Use the patterns shown here in your conversational AI application
2. **Customize channels**: Define channel configurations that match your use cases
3. **Monitor in production**: Set up proper logging and monitoring
4. **Scale as needed**: Consider Redis clustering for high-volume applications
5. **Explore advanced features**: Content processing, custom storage adapters, etc.

### Resources:
- [NexaMem Documentation](https://github.com/your-repo/nexamem)
- [Azure Cache for Redis Documentation](https://docs.microsoft.com/azure/azure-cache-for-redis/)
- [Redis Best Practices](https://redis.io/topics/memory-optimization)

In [ ]:
# Optional: Cleanup demo data
print("=== Optional Cleanup ===\n")
print("To clean up all demo data created in this notebook, run this cell:")

def cleanup_demo_data():
    """Clean up all demo data created in this notebook."""
    scopes_to_clean = [
        "python_tutorial", "tutorial_sessions", "temp_scope", 
        "async_sessions", "short_term_scope", "medium_term_scope", 
        "long_term_scope", "batch_demo", "educational_sessions"
    ]
    
    total_deleted = 0
    for scope in scopes_to_clean:
        deleted = memory.clear_scope(scope)
        if deleted > 0:
            print(f"  ✓ Deleted {deleted} memories from '{scope}' scope")
            total_deleted += deleted
    
    # Clean up user-specific scopes
    for user in ["alice", "bob", "charlie"]:
        user_scope = f"user_{user}_session"
        deleted = memory.clear_scope(user_scope)
        if deleted > 0:
            print(f"  ✓ Deleted {deleted} memories from '{user_scope}' scope")
            total_deleted += deleted
    
    print(f"\n🧹 Total cleanup: {total_deleted} memories deleted")
    print("✅ Demo cleanup completed!")

# Uncomment the next line to actually perform cleanup
# cleanup_demo_data()

print("Cleanup function defined. Uncomment the last line to run cleanup.")
print("\n🎉 NexaMem + Azure Cache for Redis demo completed successfully!")

# Show final stats
final_stats = memory.get_stats()
print(f"\n📊 Final System Statistics:")
for key, value in final_stats.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:,}")
    else:
        print(f"  {key}: {value}")